In [1]:
import sys
import json
import pandas as pd

sys.path.append("..")

from etl_pipeline_local import get_over_25_table

In [2]:
# Data upload - do not implement
# Bet setup
target_col = "over_25"

# Load data
df_loaded = get_over_25_table()
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)
df = df_loaded.copy()

In [3]:
def get_mask(df, params_dict):
    mask = pd.Series(True, index=df.index)
    features = set()

    for key in params_dict:
        for suffix in ["_cat", "_use_min", "_use_max", "_min", "_max", "_include_missing"]: # ,   "_min_idx", "_max_idx", 
             if key.endswith(suffix):
                features.add(key[:-len(suffix)])
                break

    for feat in features:
        s = df[feat]

        include_missing = params_dict.get(f"{feat}_include_missing", False)
        use_min = params_dict.get(f"{feat}_use_min", True)
        use_max = params_dict.get(f"{feat}_use_max", True)

        feat_mask = pd.Series(True, index=df.index)

        # Categorical filtering
        if f"{feat}_cat" in params_dict:
            feat_mask &= s.isin(params_dict[f"{feat}_cat"])

        # Numerical filtering
        if f"{feat}_min" in params_dict and use_min:
            feat_mask &= s >= params_dict[f"{feat}_min"]

        if f"{feat}_max" in params_dict and use_max:
            feat_mask &= s <= params_dict[f"{feat}_max"]

        if include_missing:
            feat_mask = feat_mask | s.isna()
        else:
            feat_mask = feat_mask & s.notna()

        if params_dict[f"use_{feat}"]:
            mask &= feat_mask

    return mask


def round_to_step(x, step):
    """
    Arrotonda x a un multiplo di 'step'.
    """
    try:
        if not step:
            return str(x)
        elif step <= 0:
            raise ValueError("step deve essere > 0")
        else:
            ratio = x / step
            return round(ratio) * step
    
    except Exception as e: 
        return None

In [4]:
with open("../../strategies/strategies.json", "r", encoding="utf-8") as f:
    data = json.load(f)

params_dict = data[target_col]['params_dict']
feature_bins_map = data[target_col]['feature_bins_map']

# Feature_engineering
df = df[df["underOver_quote_currentO"] >= 1.4]

df_binned = df.copy()

for feat, step in feature_bins_map.items():
    df[feat] = [round_to_step(x, step) for x in df[feat]]

    if isinstance(step, int):
        df[feat] = df[feat].astype("Int64")


mask = get_mask(df, params_dict)

df_filtered = df[mask]
df_filtered.head()

,time,chance1x2_chance_p1,chance1x2_chance_px,chance1x2_chance_p2,chance1x2_chance_p1x,chance1x2_chance_p2x,chance1x2_chance_p12,chance1x2_chance_pHt1,chance1x2_chance_pHtx,chance1x2_chance_pHt2,...,underOver_flashback_under35,underOver_flashback_over35,team_goal,team_goalHt,team_corner,chance1x2_xg_home,chance1x2_xg_away,chance1x2_xg_total,chance1x2_xg,over_25
24,2025-05-16 20:00:00,59.0,18.2,22.8,77.2,41.0,81.8,48.0,28.7,23.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
191,2025-05-18 15:30:00,60.1,23.0,16.9,83.1,39.9,77.0,47.2,35.9,16.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
280,2025-05-19 21:00:00,43.3,26.6,30.1,69.9,56.7,73.4,35.0,34.4,30.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
391,2025-05-24 18:30:00,66.1,20.5,13.4,86.6,33.9,79.5,43.9,43.6,12.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
696,2025-06-21 01:00:00,48.8,27.1,24.1,75.9,51.2,72.9,36.4,40.1,23.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True


In [5]:
df_filtered.shape

(123, 139)